# 01 · Data and ToT baseline

**Mini-crossword solver** (Avery Mattoon, Yutong Hu). CS 557 final project.

This notebook:
1. loads the ToT mini-crossword dataset and our fixed splits,
2. reproduces **ToT + GPT-4** from the trajectories the ToT authors released (free, no API calls) and checks our metric code against the paper,
3. re-runs the **ToT DFS baseline** with a model we choose, logging LLM calls, tokens and cost per puzzle,
4. saves results to `results/` so the other teammate can use them without re-running.

All real code lives in `src/xword/`. Keep this notebook thin: if you change how something works, change it in `src/` so both of us get it.

**Runs on Colab or locally.** On Colab, add two secrets (🔑 icon in the left bar) and grant this notebook access: `OPENAI_API_KEY`, and `GITHUB_TOKEN` if the repo is private.

## 0 · Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/OWNER/REPO.git"   # TODO: set to our repo once, then commit
IN_COLAB = "google.colab" in sys.modules

def secret(name):
    """Colab secret, else environment variable / .env file."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

if IN_COLAB:
    token = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    if not Path("/content/repo").exists():
        subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    # running locally from notebooks/: work from the repo root
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    try:
        from dotenv import load_dotenv; load_dotenv()   # reads OPENAI_API_KEY etc. from .env
    except ImportError:
        pass

sys.path.insert(0, str(Path("src").resolve()))
if secret("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = secret("OPENAI_API_KEY")

# Optional: keep the LLM cache on Google Drive so it survives Colab restarts.
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["XWORD_CACHE"] = "/content/drive/MyDrive/xword/llm_cache.sqlite"

print("repo root:", Path.cwd())
print("OpenAI key found:", bool(os.environ.get("OPENAI_API_KEY")))

## 1 · Data and splits

In [ ]:
import pandas as pd
from xword import load_puzzles, get_split, TEST_IDS, FEWSHOT_IDS, DEV_IDS

puzzles = load_puzzles()
pd.DataFrame({
    "split": ["test", "fewshot", "dev"],
    "puzzles": [len(TEST_IDS), len(FEWSHOT_IDS), len(DEV_IDS)],
    "use": ["final numbers only (same 20 as ToT)",
            "worked examples inside ToT's prompts: never tune or test on these",
            "tuning k, prompts, search settings"],
})

In [ ]:
print(puzzles[TEST_IDS[0]].render())

Sanity check on the metric: the gold grid should score 100% and an empty grid 0%.

In [ ]:
from xword import score_board
p = puzzles[0]
print("gold :", score_board(list(p.solution), p.solution))
print("empty:", score_board(["_"] * 25, p.solution))

## 2 · ToT + GPT-4, from the paper's released trajectories

The ToT authors released every DFS trajectory from the paper (`data/tot_logs/`). Replaying them with our code should give the paper's Table 3: **ToT 78.0 / 60.0 / 20 %** (letter / word / game), **+best state 82.4 / 67.5 / 35 %**, **−prune 65.4 / 41.5 / 5 %**.

The paper's output rule is the *first deepest* state the DFS reached. "+best state" picks the state with the best word accuracy, which uses the answer key, so it is only an oracle upper bound.

These logs have no token counts, so section 3 re-runs ToT ourselves to measure cost.

In [ ]:
from xword.experiment import tot_paper_records, summarize

paper = pd.DataFrame([
    {"method": "ToT + GPT-4 (paper logs)", **summarize(tot_paper_records(prune=True))},
    {"method": "ToT + GPT-4, no pruning (paper logs)", **summarize(tot_paper_records(prune=False))},
]).set_index("method")
paper

In [ ]:
per_puzzle = pd.DataFrame([{"idx": r["idx"], **r["scores"], "nodes": r["nodes"]}
                           for r in tot_paper_records(prune=True)]).set_index("idx")
per_puzzle.T.round(2)

## 3 · Re-run the ToT DFS baseline with our model

Same algorithm and prompts as ToT (`src/xword/tot.py`, checked state-for-state against ToT's original notebook), with every call counted.

**Choose the model here.**
* `BACKEND = "openai"`: any OpenAI-compatible API (`gpt-4o`, `gpt-4o-mini`, …). For an open model served elsewhere, set `OPENAI_BASE_URL`.
* `BACKEND = "hf"`: a local Hugging Face model on the Colab GPU, e.g. `meta-llama/Llama-3.1-8B-Instruct` (needs `pip install -e .[hf]` and a GPU).
* With no API key the cell falls back to a **scripted dry run** (a fake model that knows the answers). It checks the pipeline end to end and costs nothing. Its scores mean nothing.

Every response is cached in `cache/llm_cache.sqlite`, so re-running a cell replays the same answers for free.

In [ ]:
BACKEND = "openai"
MODEL = "gpt-4o-mini"      # cheap for testing; switch to "gpt-4o" etc. for real runs
SPLIT = "test"

import re
from xword import SLOTS
from xword.llm import make_llm, ScriptedLLM
from xword.tot import ToTSolver, ToTConfig

DRY_RUN = BACKEND == "openai" and not os.environ.get("OPENAI_API_KEY")

def oracle_fn_for(puzzle):
    gold = dict(zip(SLOTS, puzzle.answers))
    def fn(prompt, n):
        if prompt.startswith("Evaluate if there exists"):
            return ["sure"] * n
        unfilled = re.findall(r"^([hv][1-5])\. .*: _____$", prompt, flags=re.M)
        return ["\n".join(f"{s}. {gold[s].lower()} (certain)" for s in unfilled)] * n
    return fn

if DRY_RUN:
    print("No OPENAI_API_KEY: dry run with a scripted model (scores are meaningless).")
    llm = ScriptedLLM(oracle_fn_for(get_split(SPLIT)[0]), model="scripted-oracle")
else:
    llm = make_llm(BACKEND, MODEL)
print("model:", llm.model)

### 3a · Smoke test: one puzzle, small search budget

Run this first. It shows real model output and projects the cost of the full run before you commit to it.

In [ ]:
smoke_cfg = ToTConfig(time_limit=10)          # ToT uses 100; 10 keeps this quick
solver = ToTSolver(llm, smoke_cfg, verbose=True)
rec = solver.solve(get_split(SPLIT)[0])

print("\nfinal board:", rec["board"], " scores:", rec["scores"])
print("usage:", rec["usage"])

# rough projection: full run = ~100 nodes/puzzle x 20 puzzles
if rec["nodes"]:
    from xword.llm import PRICES_PER_M
    scale = (100 / rec["nodes"]) * len(get_split(SPLIT))
    tok_in, tok_out = rec["usage"]["prompt_tokens"] * scale, rec["usage"]["completion_tokens"] * scale
    print(f"projected full {SPLIT} run: ~{tok_in/1e6:.1f}M input + ~{tok_out/1e6:.1f}M output tokens (upper bound)")
    if llm.model in PRICES_PER_M:
        pin, pout = PRICES_PER_M[llm.model]
        print(f"≈ ${(tok_in * pin + tok_out * pout) / 1e6:.2f} at list price for {llm.model} (check current pricing)")

### 3b · Full baseline run

Set `RUN_FULL = True` to run all puzzles in the split with ToT's settings (time limit 100, prune, 3 children per state, 8 proposal samples).

Results are appended to `results/tot_dfs__<model>__<split>.jsonl` after **each** puzzle. If Colab disconnects, re-run this cell and it picks up where it stopped.

In [ ]:
RUN_FULL = False

from xword.experiment import run, results_path, load_results

out_path = results_path("tot_dfs", llm.model, SPLIT)
if RUN_FULL and not DRY_RUN:
    records = run(ToTSolver(llm, ToTConfig()), get_split(SPLIT), out_path)
else:
    records = load_results(out_path)
    print(f"RUN_FULL is off (or dry run); loaded {len(records)} saved records from {out_path.name}")
print("billed this session:", llm.usage.billed_calls, "calls,",
      llm.usage.billed_prompt_tokens + llm.usage.billed_completion_tokens, "tokens")

## 4 · Results so far

Every results file in `results/` (yours and your teammate's) next to the paper's GPT-4 numbers. Accuracy uses ToT's rule; `oracle_*` is the "+best state" upper bound. Cost columns are per puzzle and count cached calls too, i.e. what the method costs, not what this session paid.

In [ ]:
from pathlib import Path
rows = [{"method": "ToT + GPT-4 (paper logs)", **summarize(tot_paper_records(prune=True))}]
for f in sorted(Path("results").glob("*.jsonl")):
    recs = load_results(f)
    if recs:
        rows.append({"method": f.stem, **summarize(recs)})
summary = pd.DataFrame(rows).set_index("method")
Path("results").mkdir(exist_ok=True)
summary.to_csv("results/summary.csv")
summary

## 5 · Share results (Colab)

Results only reach your teammate once they are pushed. Locally, commit `results/` as usual. On Colab, fill in your name and email and set `PUSH = True`. This needs a `GITHUB_TOKEN` secret with write access.

In [ ]:
PUSH = False
GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"

if PUSH and IN_COLAB:
    for cmd in (["git", "config", "user.name", GIT_NAME], ["git", "config", "user.email", GIT_EMAIL],
                ["git", "pull", "-q", "--rebase", "--autostash"], ["git", "add", "results/"],
                ["git", "commit", "-q", "-m", f"Add ToT baseline results ({llm.model}, {SPLIT})"],
                ["git", "push", "-q"]):
        print(" ".join(cmd[:3]), "->", subprocess.run(cmd).returncode)